# ILCOF-1 from SMILES to powder diffraction

Build a square imine-linked COF, compare two ways its layers can stack and inspect their powder X-ray diffraction (PXRD) patterns against experiment. Open this notebook in its own folder. Use a Python 3.12 kernel with the notebook and reaction extras installed from this checkout (`python -m pip install '.[notebook,reactions]'` from the repository root).

In [ ]:
import coflandscaper as cl

## Define the framework

ILCOF-1 pairs a four-connected pyrene tetraaldehyde with p-phenylenediamine. These are the molecular precursors for the square (`sql`) net. [ILCOF-1 synthesis](https://doi.org/10.1002/chem.201203753)

In [ ]:
COF_NAME = "ILCOF-1"
TOPOLOGY = "sql"
NODE_SMILES = "O=Cc1ccc(-c2cc(-c3ccc(C=O)cc3)c3ccc4c(-c5ccc(C=O)cc5)cc(-c5ccc(C=O)cc5)c5ccc2c3c54)cc1"
LINKER_SMILES = "Nc1ccc(N)cc1"

## Build one layer

The builder prepares the precursors in `0_node/ILCOF-1/` and `0_linker/ILCOF-1/`, assembles the layer and checks the E-imine junctions. The initial CIF goes into `ILCOF-1/1_ILCOF-1_single_layer/`.

In [ ]:
cl.BuildCOF2D().build(
    TOPOLOGY, COF_NAME, node_smiles=NODE_SMILES, linker_smiles=LINKER_SMILES
)

## Relax the single layer

The default MACE-MH-1 calculator relaxes the in-plane cell lengths independently, holding the right angles and 15 Å cell height fixed. Its first use may download the model. A converged layer is saved as `ILCOF-1_preopt.cif` in stage 1. If convergence fails, this cell stops and reports the saved intermediate.

In [ ]:
opt = cl.GeometryOpt()
opt.run_preopt(COF_NAME, topo=TOPOLOGY, strict=True)

## Sample layer stacking

The introductory grid pairs layer separations of 3.4 and 3.6 Å with lateral slips of 0, 0.5 and 1.0 Å along the square-cell diagonal. Zero slip is AA stacking. Inclined and serrated stacks give twelve CIFs in `ILCOF-1/2_ILCOF-1_matrix/`. This coarse grid illustrates the method. Establishing a global minimum requires a wider, converged search.

In [ ]:
cl.CreateMatrix.quick_start(COF_NAME, TOPOLOGY)

## Calculate single-point energies

Evaluate each stacked geometry with the same calculator used for the layer. Each mode gets its own energy table in `ILCOF-1/3_ILCOF-1_landscape/`.

In [ ]:
opt.single_point.run(COF_NAME)

## View the landscapes

The figures show how relative energy changes with separation and slip. Each star marks the lowest sampled energy for that mode. The figures are also saved in stage 3.

In [ ]:
cl.Landscape().run(COF_NAME)

## Select candidates

Choose the lowest sampled energy in each mode for relaxation. The selected CIFs go into the stage 3 `selection/` folder.

In [ ]:
cl.SelectCofs().run(COF_NAME)

## Relax the stacked candidates

The atoms and full cell can now relax. This is usually the longest CPU step. Stage 4 contains the resulting CIFs and per-layer energies. In the returned summary, `True` indicates convergence and `False` marks a provisional geometry.

In [ ]:
opt.run(COF_NAME)

## Compare the final geometries

Inspect the interlayer distance (ILD, Å), lateral slip (ILS, Å) and energies of each candidate. The table is also saved as `ILCOF-1/5_ILCOF-1_analysis/final_structures.csv`.

In [ ]:
cl.AnalyzeStacking().analyze(COF_NAME)

## Compare diffraction patterns

The supplied experimental scan uses a Cu wavelength of 1.5406 Å. Simulate with the matching Cu Kα₁ preset. Stage 5 stores the simulated patterns under `pxrd_xy/`.

In [ ]:
pxrd = cl.PXRD(wavelength="CuKa1")
pxrd.run(COF_NAME)

Compare the simulated reflections with the measured trace from `experimental_pxrd/`. Each candidate gets a comparison figure under `pxrd_plots/` in stage 5.

In [ ]:
pxrd.plot_sim_vs_exp(COF_NAME)